# Data Cleaning : Healthcare Assessment

Welcome! In this notebook, we will  clean "dirty" data using Python's **Pandas** library. Data cleaning is often 80% of a data scientist's job, so mastering these skills is crucial.

## Objectives:
1. **Exploration**: Identifying missing values, duplicates, and outliers.
2. **Handling Missing Data**: Filling or dropping null values.
3. **Type Conversion**: Ensuring numeric data is in the correct format.
4. **Sanity Checks**: Removing illogical values (like negative physical activity).
5. **Feature Engineering (Basics)**: Cleaning up noise.

### Step 1: Loading the Data
First, we import `pandas` and load our raw CSV file.

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = '../data/raw/dirty_v3_path.csv'
df = pd.read_csv(file_path)

# Show the first few rows
df.head()

,Age,Gender,Medical Condition,Glucose,Blood Pressure,BMI,Oxygen Saturation,LengthOfStay,Cholesterol,Triglycerides,HbA1c,Smoking,Alcohol,Physical Activity,Diet Score,Family History,Stress Level,Sleep Hours,random_notes,noise_col
0,46.0,Male,Diabetes,137.04,135.27,28.90,96.04,6,231.88,210.56,7.61,0,0,-0.20,3.54,0,5.07,6.05,lorem,-137.057211
1,22.0,Male,Healthy,71.58,113.27,26.29,97.54,2,165.57,129.41,4.91,0,0,8.12,5.90,0,5.87,7.72,ipsum,-11.230610
2,50.0,NaN,Asthma,95.24,NaN,22.53,90.31,2,214.94,165.35,5.60,0,0,5.01,4.65,1,3.09,4.82,ipsum,98.331195
3,57.0,NaN,Obesity,NaN,130.53,38.47,96.60,5,197.71,182.13,6.92,0,0,3.16,3.37,0,3.01,5.33,lorem,44.187175
4,66.0,Female,Hypertension,95.15,178.17,31.12,94.90,4,259.53,115.85,5.98,0,1,3.56,3.40,0,6.38,6.64,lorem,44.831426


### Step 2: Inspection
How "dirty" is our data? We use `df.info()` and `df.isnull().sum()` to find out.

In [2]:
print("--- Dataset Info ---")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum())

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Age                25500 non-null  float64
 1   Gender             25500 non-null  object 
 2   Medical Condition  25500 non-null  object 
 3   Glucose            25500 non-null  float64
 4   Blood Pressure     25500 non-null  float64
 5   BMI                30000 non-null  float64
 6   Oxygen Saturation  30000 non-null  float64
 7   LengthOfStay       30000 non-null  int64  
 8   Cholesterol        30000 non-null  float64
 9   Triglycerides      30000 non-null  float64
 10  HbA1c              30000 non-null  float64
 11  Smoking            30000 non-null  int64  
 12  Alcohol            30000 non-null  int64  
 13  Physical Activity  30000 non-null  float64
 14  Diet Score         30000 non-null  float64
 15  Family History     30000 non-null  int64  
 16  S

### Step 3: Handling Missing Values
We have a few strategies:
- **Imputation**: Fill with Mean (for numbers) or Mode (for categories).
- **Dropping**: Remove rows if the data is too critical to guess.

In this case, we'll fill `Gender` with 'Unknown' and `Glucose` with the median value.

In [3]:
# Fill categorical missing values
df['Gender'] = df['Gender'].fillna('Unknown')
df['Medical Condition'] = df['Medical Condition'].fillna('Unknown')

# Fill numeric missing values with median (robust to outliers)
df['Glucose'] = df['Glucose'].fillna(df['Glucose'].median())
df['Blood Pressure'] = df['Blood Pressure'].fillna(df['Blood Pressure'].median())

print("Missing values after cleaning:")
print(df[['Gender', 'Glucose', 'Blood Pressure']].isnull().sum())

Missing values after cleaning:
Gender            0
Glucose           0
Blood Pressure    0
dtype: int64


### Step 4: Data Type Correction & Outliers
Some values might be logically impossible. 
1. **Age**: Should be an integer.
2. **Physical Activity**: Cannot be negative.
3. **Noise**: The column `noise_col` seems useless, so we'll drop it.

In [4]:
# 1. Convert Age to integer (after filling any NaNs)
df['Age'] = df['Age'].fillna(df['Age'].median()).astype(int)

# 2. Fix negative Physical Activity (take absolute value or cap at 0)
df['Physical Activity'] = df['Physical Activity'].apply(lambda x: max(0, x))

# 3. Drop useless columns
cols_to_drop = ['noise_col', 'random_notes']
df_cleaned = df.drop(columns=cols_to_drop)

df_cleaned.head()

,Age,Gender,Medical Condition,Glucose,Blood Pressure,BMI,Oxygen Saturation,LengthOfStay,Cholesterol,Triglycerides,HbA1c,Smoking,Alcohol,Physical Activity,Diet Score,Family History,Stress Level,Sleep Hours
0,46,Male,Diabetes,137.04,135.27,28.90,96.04,6,231.88,210.56,7.61,0,0,0.00,3.54,0,5.07,6.05
1,22,Male,Healthy,71.58,113.27,26.29,97.54,2,165.57,129.41,4.91,0,0,8.12,5.90,0,5.87,7.72
2,50,Unknown,Asthma,95.24,138.32,22.53,90.31,2,214.94,165.35,5.60,0,0,5.01,4.65,1,3.09,4.82
3,57,Unknown,Obesity,110.50,130.53,38.47,96.60,5,197.71,182.13,6.92,0,0,3.16,3.37,0,3.01,5.33
4,66,Female,Hypertension,95.15,178.17,31.12,94.90,4,259.53,115.85,5.98,0,1,3.56,3.40,0,6.38,6.64


### Step 5: Saving Cleaned Data
Finally, we save our hard work to the `processed` folder.

In [5]:
output_path = '../data/processed/cleaned_healthcare_data.csv'
df_cleaned.to_csv(output_path, index=False)
print(f"Cleaned data saved to: {output_path}")

Cleaned data saved to: ../data/processed/cleaned_healthcare_data.csv
